In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys
from tqdm import tqdm

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

from roi_classifier.prepare_data import prepare_roi_data
from roi_classifier.annotate_data import annotate_rois
from roi_classifier.train_classifier import train_roi_classifier




In [2]:
DATASET_ROOT = Path(r"C:\Users\mzinn1\Desktop\invivo_tiffs")  # TODO: set this to your data path
assert DATASET_ROOT.exists(), f"Dataset root {DATASET_ROOT} does not exist."

ROI_DIR = PROJECT_ROOT / "data"
ROI_DIR.mkdir(parents=True, exist_ok=True)

ROI_DATA_PATH = ROI_DIR / "invivo_roi_features_z.npy"

MODEL_OUT_DIR = PROJECT_ROOT / "models"
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = PROJECT_ROOT / "config/classifier_config.yaml"

print(f"Extracting fluorescence data from {DATASET_ROOT.__str__()}")
print(f"Saving engineered data to {ROI_DATA_PATH.__str__()}")
print(f"Saving models to {MODEL_OUT_DIR.__str__()}")
print(f"Configuring classifier according to {CONFIG_PATH.__str__()}")

Extracting fluorescence data from C:\Users\mzinn1\Desktop\invivo_tiffs
Saving engineered data to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\invivo_roi_features_z.npy
Saving models to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models
Configuring classifier according to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\classifier_config.yaml


In [3]:
update = True # Change to false 
backup = False # Change as you wish; controls whether or not a backup of the original engineered data is saved
fs = 3.0       # Frame rate in Hz — set to match your acquisition rate (scales smoothing accordingly)

roi_data = prepare_roi_data(
    dataset_root=DATASET_ROOT,
    input_file=ROI_DATA_PATH,
    output_file=ROI_DATA_PATH,
    update=update,
    backup=backup,
    fs=fs
)



  ROI Summary
  Total rois: 1942
  Good: 133 | Bad: 368 | Unlabeled: 1441
  Manual: 500 | Auto: 1
  Total spikes stored: 5301

  Re-smoothing with sigma=0.80 (fs=3.0, base sigma=4.0)
  - ROI 5729L-10_0: max=1.71, min=-2.12
  - ROI 5729L-10_1: max=0.99, min=-4.12
  - ROI 5729L-10_2: max=2.09, min=-1.62
  - ROI 5729L-10_3: max=1.98, min=-2.16
  - ROI 5729L-10_4: max=2.44, min=-2.04
  - ROI 5729L-10_5: max=1.59, min=-1.58
  - ROI 5729L-10_6: max=1.82, min=-1.95
  - ROI 5729L-10_7: max=2.07, min=-2.11
  - ROI 5729L-10_8: max=1.79, min=-1.47
  - ROI 5729L-10_9: max=2.07, min=-2.24
  - ROI 5729L-10_10: max=2.32, min=-1.38
  - ROI 5729L-10_11: max=3.34, min=-1.84
  - ROI 5729L-10_12: max=2.18, min=-1.77
  - ROI 5729L-10_13: max=1.88, min=-2.13
  - ROI 5729L-10_14: max=1.99, min=-2.36
  - ROI 5729L-10_15: max=1.43, min=-3.04
  - ROI 5729L-10_16: max=1.72, min=-1.95
  - ROI 5729L-10_17: max=1.70, min=-1.83
  - ROI 5729L-10_18: max=1.65, min=-1.77
  - ROI 5729L-10_19: max=1.75, min=-2.00
  - RO

In [4]:
# Change these flags to control which ROIs are shown for annotation and how many
unlabeled_only = False 
labeled_only = True
n_samples = 1000

assert not (unlabeled_only and labeled_only), "unlabeled_only and labeled_only cannot both be True — pick one or set both to False to show all ROIs."

annotate_rois(data_path=ROI_DATA_PATH,
              n_samples=n_samples,
              unlabeled_only=unlabeled_only,
              labeled_only=labeled_only)

Found 501 roi keys matching filter
Returning all 501 keys.
Session ended by user. Saving progress...
Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\invivo_roi_features_z.npy
Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\invivo_roi_features_z.npy

  ROI Annotation Summary
  Queued:    501
  Seen:      0
  Labeled:   0
  Updated:   0
  Confirmed: 0
  Skipped:   0


  ROI Summary
  Total rois: 1942
  Good: 133 | Bad: 368 | Unlabeled: 1441
  Manual: 500 | Auto: 1
  Total spikes stored: 5301



{'level': 'roi',
 'queued': 501,
 'total': 0,
 'labeled': 0,
 'updated': 0,
 'confirmed': 0,
 'skipped': 0}

In [5]:
name = "z_invivo_roi_classifier" # TODO Change this as needed for your own experimental/organizational needs
data_paths = [ROI_DATA_PATH] # Can be a list of paths if you have engineered data from multiple sources you want to combine for training
results = train_roi_classifier(config_path=CONFIG_PATH, data_path=data_paths, name=name,
                     output_dir=MODEL_OUT_DIR, verbose=True, manual_only=True, overwrite=False)

Dataset Summary
--------------------------------------------------
Total labeled datapoints: 500
  Train: 400 | Test: 100

Label distribution:
              Bad (0)  Good (1)
  Train           296       104
  Test             71        29
  Total           367       133

Training on: Manual labels only


c:\Users\mzinn1\AppData\Local\anaconda3\envs\gcamp\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



--------------------------------------------------
TUNED MODEL SUMMARY
--------------------------------------------------
Model:     RandomForestClassifier
Transform: raw
Features:  ['absolute_derivative_skew', 'derivative_skew', 'var_of_var', 'spike_prom_skew', 'peak_density', 'median_spike_prom', 'ac_decay', 'w_peak_density', 'snr_estimate', 'derivative_asymmetry', 'range_trace', 'spike_prom_mean']

Hyperparameters:
  class_weight: balanced
  max_depth: 5
  min_samples_leaf: 4
  min_samples_split: 2
  n_estimators: 100

Metrics:
  CV Accuracy:   0.9400
  Test Accuracy: 0.9500
  ROC AUC:       0.9898
  F1:            0.9492
  Precision:     0.9505
  Recall:        0.9500

Confusion Matrix:
              Pred 0  Pred 1
  Actual 0    70      1      
  Actual 1    4       25     
--------------------------------------------------
Saved model to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models\z_invivo_roi_classifier.joblib
Saved results to C:\Users\mzinn1\Desktop\Scripts\GCaMP-anal